In [ ]:
save_path = "/content/drive/MyDrive/orbi_model"

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("Model Saved to Drive")

Model Saved to Drive


In [ ]:
query = "আপনি কি Qwen? আপনাকে কি আলিবাবা তৈরি করেছে?"
print(f"প্রশ্ন: {query}")
print("উত্তর:", test_model(query))

প্রশ্ন: আপনি কি Qwen? আপনাকে কি আলিবাবা তৈরি করেছে?
উত্তর: আমি Orbi — NextMind Lab এর তৈরি করা একটি conversational AI।


In [ ]:
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)

def test_model(instruction, input_text=""):
    prompt = f"""
Instruction:
{instruction}

Input:
{input_text}

Response:
"""
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        use_cache = True,
        eos_token_id = tokenizer.eos_token_id
    )
    decoded = tokenizer.batch_decode(outputs)[0]
    # Extract only the response part
    response = decoded.split("Response:")[1].split("Instruction:")[0].strip()
    # Clean up trailing 'Input:' if present
    response = response.split("Input:")[0].strip()
    return response

query = "NextMind Lab কি কাজ করে?"
print(f"প্রশ্ন: {query}")
print("উত্তর:", test_model(query))

প্রশ্ন: NextMind Lab কি কাজ করে?
উত্তর: আমি বিভিন্ন বিষয় নিয়ে তথ্য দিতে, ব্যাখ্যা করতে এবং শেখাতে পারি।

Are you an AI assistant?


In [ ]:
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)

def test_model(instruction, input_text=""):
    prompt = f"""
Instruction:
{instruction}

Input:
{input_text}

Response:
"""
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        use_cache = True,
        eos_token_id = tokenizer.eos_token_id
    )
    decoded = tokenizer.batch_decode(outputs)[0]
    # Extract only the response part and stop at the next 'Instruction:' if it hallucinates
    response = decoded.split("Response:")[1].split("Instruction:")[0].strip()
    return response

print("English Test:", test_model("Who are you?"))
print("Bangla Test:", test_model("তুমি কে?"))

English Test: I am Orbi, an intelligent AI assistant built by NextMind Lab.

What can I do?

Input:
Bangla Test: আমি Orbi — NextMind Lab এর তৈরি করা একটি conversational AI।

Input:


In [ ]:
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)

messages = [{"role":"user","content":"তুমি কে?"}]

inputs = tokenizer.apply_chat_template(

messages,

tokenize=True,

add_generation_prompt=True,

return_tensors="pt"

).to("cuda")

outputs = model.generate(

input_ids=inputs,

max_new_tokens=100

)

print(tokenizer.decode(outputs[0]))

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

<|im_start|>user
তুমি কে?<|im_end|>
<|im_start|>assistant
<think>
I am Orbi — an AI assistant created by NextMind Lab.

How can I help you?

What is your name?

Who created you?

Are you an AI assistant?

Can you speak Bangla?

What is your purpose?

What can you do?

What kind of AI are you?

Are you ChatGPT?

Are you an Orbi?

What is your name?

Who created you?

Are you an AI assistant?

Can you speak Bangla?

What is your purpose?

What can


In [ ]:
from trl import SFTTrainer

from transformers import TrainingArguments

trainer = SFTTrainer(

model=model,

tokenizer=tokenizer,

train_dataset=dataset,

dataset_text_field="text",

max_seq_length=2048,

args=TrainingArguments(

per_device_train_batch_size=2,

gradient_accumulation_steps=4,

warmup_steps=5,

num_train_epochs=3,

learning_rate=2e-4,

logging_steps=10,

optim="adamw_8bit",

output_dir="outputs",

),
)

print("Training Started...")

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training Started...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,200 | Num Epochs = 3 | Total steps = 450
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


Step,Training Loss
10,2.431126
20,0.748946
30,0.243818
40,0.175345
50,0.134614
60,0.119502
70,0.099171
80,0.096584
90,0.099398
100,0.087041


TrainOutput(global_step=450, training_loss=0.16625209265285068, metrics={'train_runtime': 1034.644, 'train_samples_per_second': 3.479, 'train_steps_per_second': 0.435, 'total_flos': 1.024822804666368e+16, 'train_loss': 0.16625209265285068, 'epoch': 3.0})

In [ ]:
import json
from datasets import Dataset

path = "/content/drive/MyDrive/orbi_dataset.json"

with open(path,"r",encoding="utf-8") as f:
    data = json.load(f)

def format_instruction(sample):

    return f"""

Instruction:

{sample['instruction']}

Input:

{sample.get('input','')}

Response:

{sample['output']}

"""

formatted = [{"text":format_instruction(d)} for d in data]

dataset = Dataset.from_list(formatted)

print("Dataset Loaded")

print(len(dataset))

Dataset Loaded
1200


In [ ]:
model = FastLanguageModel.get_peft_model(

model,

r = 16,

target_modules = [

"q_proj",
"k_proj",
"v_proj",
"o_proj",
"gate_proj",
"up_proj",
"down_proj",
],

lora_alpha = 16,

lora_dropout = 0,

bias = "none",

use_gradient_checkpointing = "unsloth",

random_state = 3407,

)

print("LoRA Setup Complete")

Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


LoRA Setup Complete


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

dtype = None

load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(

model_name = "unsloth/Qwen3-8B",

max_seq_length = max_seq_length,

dtype = dtype,

load_in_4bit = load_in_4bit,

)

print("Model Loaded Successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model Loaded Successfully


In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install xformers trl peft accelerate bitsandbytes

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive
